## Step 1: Connect to S3
Loads the bucket name from `.env` and creates a boto3 S3 client.

In [1]:
import os
from dotenv import load_dotenv
import boto3

load_dotenv()
bucket_name = os.getenv("AWS_S3_BUCKET")
print("Bucket:", bucket_name)

s3 = boto3.client("s3", region_name="ap-southeast-2")

Bucket: pl-analytics-2026


## Step 2: Upload processed and feature datasets

In [2]:
files_to_upload_v2 = {
    "../data/processed/clean_matches_multi_season.csv": "processed/clean_matches_multi_season/clean_matches_multi_season.csv",
    "../data/features/match_features_multi_season.csv": "features/match_features_multi_season/match_features_multi_season.csv",
}

for local_path, s3_key in files_to_upload_v2.items():
    s3.upload_file(local_path, bucket_name, s3_key)
    print(f"Uploaded {local_path} -> s3://{bucket_name}/{s3_key}")

Uploaded ../data/processed/clean_matches_multi_season.csv -> s3://pl-analytics-2026/processed/clean_matches_multi_season/clean_matches_multi_season.csv
Uploaded ../data/features/match_features_multi_season.csv -> s3://pl-analytics-2026/features/match_features_multi_season/match_features_multi_season.csv


## Step 3: List everything currently in the bucket
Used here to spot inconsistent folder structure — note the mix of flat files (`processed/clean_matches.csv`) and subfoldered ones (`processed/clean_matches_multi_season/...`), which gets cleaned up in the cells below.

In [3]:
response = s3.list_objects_v2(Bucket=bucket_name)
for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes")

features/match_features.csv 109650 bytes
features/match_features_multi_season.csv 574948 bytes
features/match_features_multi_season/match_features_multi_season.csv 369238 bytes
predictions/predictions/predictions.csv 11238 bytes
processed/clean_matches/clean_matches.csv 37617 bytes
processed/clean_matches_multi_season/clean_matches_multi_season.csv 118953 bytes
raw/raw_matches_incremental/2026-09-06/matches.csv 896 bytes
raw/raw_matches_incremental/2026-09-07/matches.csv 1075 bytes
raw/raw_matches_incremental/2026-09-08/matches.csv 980 bytes
raw/raw_matches_incremental/2026-09-09/matches.csv 299 bytes
raw/raw_matches_incremental/2026-09-13/matches.csv 767 bytes
raw/raw_matches_multi_season.csv 1087039 bytes
raw/raw_standings.csv 3556 bytes
raw/raw_teams.csv 91524 bytes
raw/raw_teams_multi_season.csv 119052 bytes


## Sanity check: confirm column types match expectations
Read both uploaded CSVs back locally and print their dtypes before trusting them for downstream Glue/Athena tables.

In [4]:
import pandas as pd

clean_matches_multi = pd.read_csv("../data/processed/clean_matches_multi_season.csv")
for col in clean_matches_multi.columns:
    print(col, clean_matches_multi[col].dtype)

print("---")

match_features_multi = pd.read_csv("../data/features/match_features_multi_season.csv")
for col in match_features_multi.columns:
    print(col, match_features_multi[col].dtype)

id int64
utcDate str
status str
matchday int64
season int64
home_team_id int64
home_team_name str
away_team_id int64
away_team_name str
home_goals int64
away_goals int64
winner str
---
utcDate str
matchday int64
season int64
match_id int64
team str
goals_scored int64
goals_conceded int64
is_home bool
result str
points int64
rolling_form_pts float64
win_streak int64
loss_streak int64
avg_goals_scored float64
avg_goals_conceded float64
goal_difference int64
clean_sheet int64
clean_sheet_pct float64
days_since_last_match float64
is_win int64
home_win_pct float64
away_win_pct float64
h2h_win_pct float64
prior_season_position float64


## Step 4: Upload model predictions

In [5]:
s3.upload_file("../data/features/predictions.csv", bucket_name, "predictions/predictions/predictions.csv")

## Step 5: Remove stray flat files
Cleans up the old, inconsistent flat-file uploads so every dataset lives in its own subfolder — see Known Issue #1 in PROJECT.md/HANDOVER.md.

In [6]:
s3.delete_object(Bucket=bucket_name, Key="processed/clean_matches.csv")
s3.delete_object(Bucket=bucket_name, Key="processed/clean_matches_multi_season.csv")
print("Deleted stray files")

Deleted stray files


## Step 6: Re-upload clean_matches into its proper subfolder

In [7]:
s3.upload_file(
    "../data/processed/clean_matches.csv",
    bucket_name,
    "processed/clean_matches/clean_matches.csv"
)
print("Uploaded clean_matches to its own subfolder")

Uploaded clean_matches to its own subfolder


## Step 7: Upload season simulation results
Uploads the mid-season and preseason Monte Carlo simulation outputs (Phase 11) so they can be registered as Athena tables and used in a new Power BI "Season Outlook" dashboard.

In [8]:
prediction_files = {
    "../data/predictions/season_simulation_summary.csv": "predictions/season_simulation_summary/season_simulation_summary.csv",
    "../data/predictions/season_simulation_raw.csv": "predictions/season_simulation_raw/season_simulation_raw.csv",
    "../data/predictions/preseason_simulation_summary.csv": "predictions/preseason_simulation_summary/preseason_simulation_summary.csv",
    "../data/predictions/preseason_simulation_raw.csv": "predictions/preseason_simulation_raw/preseason_simulation_raw.csv",
    "../data/predictions/midseason_vs_preseason_comparison.csv": "predictions/midseason_vs_preseason_comparison/midseason_vs_preseason_comparison.csv",
}

for local_path, s3_key in prediction_files.items():
    s3.upload_file(local_path, bucket_name, s3_key)
    print(f"Uploaded {local_path} -> s3://{bucket_name}/{s3_key}")

Uploaded ../data/predictions/season_simulation_summary.csv -> s3://pl-analytics-2026/predictions/season_simulation_summary/season_simulation_summary.csv
Uploaded ../data/predictions/season_simulation_raw.csv -> s3://pl-analytics-2026/predictions/season_simulation_raw/season_simulation_raw.csv
Uploaded ../data/predictions/preseason_simulation_summary.csv -> s3://pl-analytics-2026/predictions/preseason_simulation_summary/preseason_simulation_summary.csv
Uploaded ../data/predictions/preseason_simulation_raw.csv -> s3://pl-analytics-2026/predictions/preseason_simulation_raw/preseason_simulation_raw.csv
Uploaded ../data/predictions/midseason_vs_preseason_comparison.csv -> s3://pl-analytics-2026/predictions/midseason_vs_preseason_comparison/midseason_vs_preseason_comparison.csv


## Step 8: Verify the folder structure is now consistent

In [9]:
response = s3.list_objects_v2(Bucket=bucket_name, Prefix="processed/")
for obj in response.get("Contents", []):
    print(obj["Key"])

processed/clean_matches/clean_matches.csv
processed/clean_matches_multi_season/clean_matches_multi_season.csv
